In [ ]:
# Instalación de la herramienta de descarga
!pip install fiftyone


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.5/934.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.1 MB/s eta 0:0

In [ ]:
#esta parte hay que cambiarla para que lo tome de los parámetros de entrada de la web
# montar el entorno de drive - esto es para que me encuentren los ficheros,
import os

from google.colab import drive
drive.mount('/content/drive')

# incorporar el nombre del proyecto
ruta_proyecto = '/content/drive/MyDrive/'
os.chdir(ruta_proyecto)
base_dir = "./fridge"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# si queremos usar un bucket en lugar de drive,

#Primero, debes dar permiso a Colab para acceder a tu proyecto de Google Cloud:
#Ve a IAM & Admin > Service Accounts.
#Crea una cuenta (ej. "colab-service-account") y dale el rol de Editor o Vertex AI User y Storage Admin.
#En la pestaña Keys, haz clic en Add Key > Create new key y selecciona el formato JSON. Se descargará un archivo a tu ordenador.
#Sube ese archivo .json a la carpeta de archivos de Cola
import os

# 1. Ruta al archivo que acabas de subir (cambiarlo)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "./miccColab.json"

# 2. Ahora ya puedes usar comandos !gcloud o !gsutil directamente
# (Nota: a veces gcloud necesita un login extra, pero las librerías de Python ya funcionarán)
project_id = "tu-proyecto-id"
!gcloud auth activate-service-account --key-file=$GOOGLE_APPLICATION_CREDENTIALS
!gcloud config set project {project_id}


#Configurar el ID de tu Proyecto
project_id = "TU_PROJECT_ID"
!gcloud config set project {project_id}

# Comandos para habilitar las APIs
# Habilitar Vertex AI API
!gcloud services enable aiplatform.googleapis.com

# Habilitar Cloud Storage API
!gcloud services enable storage.googleapis.com

# Generar el bucket
GOOGLE_CLOUD_PROJECT = project_id
base_dir = f"gs://{project_id}-fridge/"
!gsutil mb -p $GOOGLE_CLOUD_PROJECT -c standard -l us-central1 base_dir


In [ ]:

# Descargamos imágenes y etiquetas automáticamente
import fiftyone as fo
import fiftyone.zoo as foz
import os
import csv
import cv2
import shutil
import pandas as pd

from tqdm import tqdm

# 1. Definición de la lista y rutas
lista_alimentos = [
    "Sauce",           # Mayonesa y salsas en frascos
    "Container",       # Para platos preparados, tupperwares y recipientes desconocidos
# --- Lácteos y Bebidas ---
    "Juice",            # Zumos
    "Beer",             # Cervezas
    "Wine",             # Vino
    "Cream",            # Nata o cremas
    "Butter",           # Mantequilla
    "Cheese",          # Kéfir (Se asocia a lácteos similares)
     "Milk",
    "Yogurt",          # Yogures (Griegos, Bífidus, Naturales)

    # --- Verduras y Hortalizas (Vegetables) ---
    "Tomato",           # Tomates
    "Cucumber",         # Pepino
    "Broccoli",         # Brócoli
    "Zucchini",         # Calabacín
    "Bell pepper",      # Pimiento
    "Potato",           # Patatas (a veces se guardan fuera, pero es útil)
    "Cabbage",          # Repollo/Col
    "Lettuce",          # Lechuga
    "Carrot",          # Zanahorias
     "Onion",
    "Olive",           # Aceitunas

    # --- Frutas ---
    "Strawberry",       # Fresas
    "Grape",            # Uvas
    "Banana",           # Plátanos
    "Lemon",            # Limones
    "Pear",             # Peras
    "Peach",            # Melocotones
     "Orange",
     "Apple"

    # --- Proteínas y Carnes ---
    "Chicken",          # Pollo
    "Seafood",          # Mariscos/Pescado (etiqueta general)
    "Ham",              # Jamón / Embutidos
    "Beef",             # Carne de vacuno
    "Turkey",          # Pavo en lonchas (Aunque suele ser difícil, es la etiqueta oficial)
    "Fast food"        # Hamburguesas
    "Egg",             # Huevos
    "Hamburger"        # Hamburguesas

    # --- Otros ---
    "Bread",            # Pan (si se guarda en nevera)
    "Mushroom",         # Champiñones / Setas
    "Salad",            # Ensaladas preparadas
    "Jellyfish"         # (Curiosidad: Open Images la usa a veces para gelatinas)
]
csv_path = os.path.join(base_dir, "data.csv")

# Asegurar que la carpeta base existe
os.makedirs(base_dir, exist_ok=True)


# Preparar el archivo CSV (escribimos la cabecera)
with open(csv_path, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["ruta_imagen", "alimento"])

print(f"Iniciando descarga y tagueado en {base_dir}...")

# 2. Bucle recursivo/iterativo por cada alimento
for alimento in lista_alimentos:
    print(f"\n--- Procesando: {alimento} ---")

    # Crear carpeta específica para el alimento
    alimento_dir = os.path.join(base_dir, alimento)
    os.makedirs(alimento_dir, exist_ok=True)

    try:
        # Descarga del dataset para el alimento específico
        dataset = foz.load_zoo_dataset(
            "open-images-v7",
            split="validation",
            classes=[alimento],
            max_samples=10, # Ajusta según necesites
            seed=51,
            drop_existing_dataset=False # IMPORTANTE: No borrar si ya existe
        )

        # Identificar campo de etiquetas (detections o ground_truth)
        label_field = "detections" if "detections" in dataset.get_field_schema() else "ground_truth"

        # Exportar imágenes a la subcarpeta del alimento
        # Usamos el tipo ImageDirectory porque solo queremos las fotos organizadas
        # Exportar solo si la carpeta está vacía o tiene menos archivos de los esperados
        if len(os.listdir(alimento_dir)) < 5:
            dataset.export(
                export_dir=alimento_dir,
                dataset_type=fo.types.ImageDirectory,
                label_field=label_field,
            )

    # --- LÓGICA PARA EVITAR DUPLICADOS EN EL CSV ---
    # 1. Leer rutas existentes para no repetir
        rutas_existentes = set()
        if os.path.exists(csv_path):
            try:
                df_existente = pd.read_csv(csv_path)
                if not df_existente.empty:
                    rutas_existentes = set(df_existente['ruta_imagen'].tolist())
            except Exception: # Por si el CSV está vacío o corrupto
                rutas_existentes = set()

        nuevos_registros = []

    # 2. Escanear carpeta y filtrar las que no estén en el CSV
        for filename in os.listdir(alimento_dir):
          if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            ruta_completa = os.path.abspath(os.path.join(alimento_dir, filename))

            if ruta_completa not in rutas_existentes:
                nuevos_registros.append([ruta_completa, alimento])

    # 3. Guardar solo lo nuevo (modo 'a' de append)
        if nuevos_registros:
          with open(csv_path, mode='a', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerows(nuevos_registros)
          print(f"✅ Se añadieron {len(nuevos_registros)} nuevos registros de {alimento}.")
        else:
         print(f"ℹ️ No hay imágenes nuevas para {alimento} (ya registradas).")

    except Exception as e:
        print(f"❌ Error procesando {alimento}: {e}")

print(f"\n🚀 Proceso finalizado. Archivo creado en: {csv_path}")


Iniciando descarga y tagueado en ./fridge...

--- Procesando: Sauce ---


INFO:fiftyone.zoo.datasets:Downloading split 'validation' to '/root/fiftyone/open-images-v7/validation' if necessary


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/2018_04/validation/validation-images-with-rotation.csv' to '/root/fiftyone/open-images-v7/validation/metadata/image_ids.csv'


INFO:fiftyone.utils.openimages:Downloading 'https://storage.googleapis.com/openimages/v5/class-descriptions-boxable.csv' to '/root/fiftyone/open-images-v7/validation/metadata/classes.csv'


Ignoring invalid classes ['Sauce']
You can view the available classes via `fiftyone.utils.openimages.get_classes()`


...

Dataset info written to '/root/fiftyone/open-images-v7/info.json'


INFO:fiftyone.zoo.datasets:Dataset info written to '/root/fiftyone/open-images-v7/info.json'


Loading existing dataset 'open-images-v7-validation-10'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


INFO:fiftyone.zoo.datasets:Loading existing dataset 'open-images-v7-validation-10'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


✅ Se añadieron 23 nuevos registros de Jellyfish.

🚀 Proceso finalizado. Archivo creado en: ./fridge/data.csv


In [ ]:
#preprocesamos las imágenes a tamaño estándar (ej. 640x640) para que el modelo de visión las procese más rápido

import cv2
from tqdm import tqdm

def preprocesar_dataset(input_dir, output_dir, size=(640, 640)):
    """
    Escala, normaliza y limpia el dataset de imágenes.
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"🚀 Iniciando pre-procesado. Destino: {output_dir}")

    # Recorremos las carpetas de alimentos (clases)
    for clase in os.listdir(input_dir):
        ruta_clase = os.path.join(input_dir, clase)

        # Saltamos si no es una carpeta (ej. el archivo data.csv)
        if not os.path.isdir(ruta_clase):
            continue

        ruta_clase_out = os.path.join(output_dir, clase)
        os.makedirs(ruta_clase_out, exist_ok=True)

        print(f"📦 Procesando clase: {clase}")

        for img_name in tqdm(os.listdir(ruta_clase)):
            # Solo procesamos archivos de imagen
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):

              # 1. Definimos el nombre que tendrá el archivo de salida
                # (Cambiamos la extensión original a .jpg siempre)
                nombre_final = os.path.splitext(img_name)[0] + ".jpg"
                ruta_salida_final = os.path.join(ruta_clase_out, nombre_final)

                # 2. Comprobamos si ya existe para ahorrar tiempo
                if os.path.exists(ruta_salida_final):
                    continue

                try:
                    ruta_img = os.path.join(ruta_clase, img_name)

                    # 3. Carga de imagen
                    img = cv2.imread(ruta_img)
                    if img is None:
                        continue

                    # 4. Redimensionado (Resizing)
                    # INTER_AREA es ideal para reducir tamaño manteniendo calidad
                    img_res = cv2.resize(img, size, interpolation=cv2.INTER_AREA)

                    # 5. Guardado físico
                    cv2.imwrite(ruta_salida_final, img_res)


                except Exception as e:
                    print(f"⚠️ Error en {img_name}: {e}")


# --- EJECUCIÓN ---
# base_dir definido en el paso anterior (ej: "./fridge")
dataset_limpio_dir = "./fridge_preprocessed"
preprocesar_dataset(base_dir, dataset_limpio_dir)

🚀 Iniciando pre-procesado. Destino: ./fridge_preprocessed
📦 Procesando clase: Milk


100%|██████████| 39/39 [00:00<00:00, 2790.19it/s]


📦 Procesando clase: Onion


0it [00:00, ?it/s]


📦 Procesando clase: Orange


...

100%|██████████| 193/193 [00:00<00:00, 4838.30it/s]


📦 Procesando clase: Jellyfish


100%|██████████| 23/23 [00:00<00:00, 2257.64it/s]


📦 Procesando clase: __Imagenes_prueba


100%|██████████| 15/15 [00:00<00:00, 1786.03it/s]


In [ ]:
# ahora usamos la técnica de data augmentation para no tener que buscar fotos adicionales y que el modelo pueda ser más robusto
!pip install -U albumentations

import albumentations as A
from tqdm import tqdm

# Definimos las transformaciones (El "Pipeline")
# Queremos cosas que pasen de verdad en una nevera:
transform = A.Compose([
    A.HorizontalFlip(p=0.5),              # Espejo horizontal
    A.RandomBrightnessContrast(p=0.2),    # Cambios de luz (nevera abierta/cerrada)
    A.Rotate(limit=15, p=0.5),            # Foto un poco torcida
    A.Blur(blur_limit=3, p=0.1),          # Foto desenfocada por movimiento
    A.RGBShift(p=0.1),                    # Pequeños cambios de color
])

def aumentar_dataset(input_dir, num_variantes=3):
    print(f"🔄 Iniciando Augmentation en {input_dir}...")

    for clase in os.listdir(input_dir):
        ruta_clase = os.path.join(input_dir, clase)
        if not os.path.isdir(ruta_clase): continue

        print(f"✨ Aumentando clase: {clase}")

        for img_name in tqdm(os.listdir(ruta_clase)):
            if "_aug_" in img_name: continue # Evitar aumentar lo ya aumentado

            ruta_img = os.path.join(ruta_clase, img_name)
            image = cv2.imread(ruta_img)
            if image is None: continue

            # Crear N variantes
            for i in range(num_variantes):
                augmented = transform(image=image)["image"]

                # Nombre nuevo: nombreoriginal_aug_1.jpg
                nombre_base = os.path.splitext(img_name)[0]
                nuevo_nombre = f"{nombre_base}_aug_{i}.jpg"
                cv2.imwrite(os.path.join(ruta_clase, nuevo_nombre), augmented)

# Ejecución
aumentar_dataset(dataset_limpio_dir)



🔄 Iniciando Augmentation en ./fridge_preprocessed...
✨ Aumentando clase: Milk


100%|██████████| 39/39 [00:03<00:00, 11.54it/s]


✨ Aumentando clase: Onion


0it [00:00, ?it/s]


✨ Aumentando clase: Orange


...

100%|██████████| 193/193 [00:17<00:00, 10.79it/s]


✨ Aumentando clase: Jellyfish


100%|██████████| 23/23 [00:02<00:00,  8.57it/s]


✨ Aumentando clase: __Imagenes_prueba


100%|██████████| 15/15 [00:01<00:00, 10.57it/s]


In [ ]:
# obtenemos una tabla con todas las imágenes que tenemos recuperadas
import pandas as pd

def contar_imagenes(directorio):
    datos = []
    total_general = 0

    # Listar las subcarpetas (clases)
    clases = sorted([d for d in os.listdir(directorio) if os.path.isdir(os.path.join(directorio, d))])

    for clase in clases:
        ruta_clase = os.path.join(directorio, clase)
        # Contamos archivos con extensiones de imagen
        imagenes = [f for f in os.listdir(ruta_clase) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        cantidad = len(imagenes)
        datos.append({"Clase": clase, "Cantidad": cantidad})
        total_general += cantidad

    # Crear un DataFrame para mostrarlo bonito
    df_conteo = pd.DataFrame(datos)

    print(f"📊 Resumen del Dataset en: {directorio}")
    print("-" * 40)
    print(df_conteo.to_string(index=False))
    print("-" * 40)
    print(f"✅ TOTAL GENERAL: {total_general} imágenes")

    return df_conteo

# Ejecución
conteo_df = contar_imagenes("./fridge_preprocessed")

📊 Resumen del Dataset en: ./fridge_preprocessed
----------------------------------------
            Clase  Cantidad
            Apple       196
     AppleChicken         0
           Banana       128
             Beef         0
             Beer       220
      Bell pepper       184
         Broccoli       252
           Butter         0
          Cabbage       120
           Carrot       480
           Cheese       308
        Container         0
            Cream       108
         Cucumber       300
     Fast foodEgg         0
            Grape       404
              Ham         0
   HamburgerBread         0
        Jellyfish        92
            Juice       204
            Lemon       224
          Lettuce         0
             Milk       156
         Mushroom       160
            Olive         0
            Onion         0
           Orange       328
            Peach       280
             Pear       136
           Potato       248
            Salad       772
            Sau